In [78]:
# IMPORTS, CONSTANTS
import sys
import json
import pandas as pd
from pathlib import Path
from copy import deepcopy
sys.path.append(str(Path.cwd().parent))
from lib import paths

MINUTES_STAT_ID = "40"

STARTER_SLOTS = {0, 1, 2, 3, 4, 5}
BENCH_SLOTS = {6, 7, 8, 9}

MAX_MINUTES_PER_GAME = 40

path = paths.DATA / "wnba.json"

with open(path) as f:
    data = json.load(f)

metadata = data['metadata']
data = data['data']
# Print Metadata
print("Current data was taken: " + metadata['pulled_at_readable'])

Current data was taken: 2026-05-17 01:08:04 PM EDT


**data.keys() -> metadata, data**
1. data['metadata']
    - pulled_at
    - pulled_at_readable
    - league_id
    - season  
2. data['data'].keys():  
    - draftDetail  
        * drafted = True  
        * inProgress = False  
    - gameId = 5  
    - id = 1039832288   
    - members  
        * List of team dicts  
            - displayName  
            - firstName  
            - id  
            - lastName  
            - notificationSettings  
    - schedule
        * List of matchup dicts
            - away (same keys as home)
            - home
                * cumulativeScore
                    - losses
                    - scoreByStat
                        * keys 0, 1, 17, 2, 3, 6
                        * values ineligble, rank, result, score
                    - statBySlot
                    - ties
                    - wins
                * gamesPlayed
                * pointsByScoringPeriod
                    - dict with keys 1-10 for day in matchup
                * rosterForCurrentScoringPeriod
                    - appliedStatTotal
                    - entries - List of player dicts
                        * acquisitionDate
                        * acquisitionType
                        * injuryStatus
                        * lineupSlotId
                        * pendingTransactionIds
                        * playerId
                        * playerPoolEntry
                            - appliedStatTotal
                            - id
                            - keeperValue
                            - keeperValueFuture
                            - lineupLocked
                            - onTeamId
                            - player - dict of player...
                            - rosterLocked
                            - status
                            - tradeLocked
                        * status
                * rosterForMatchupPeriod
                    - appliedStatTotal (650)
                    - entries - List of player dicts
                        * acquisitionDate
                        * acquisitionType
                        * injuryStatus
                        * lineupSlotId
                        * pendingTransactionIds
                        * playerId
                        * playerPoolEntry
                            - appliedStatTotal
                            - id
                            - keeperValue
                            - keeperValueFuture
                            - lineupLocked
                            - onTeamId
                            - player - dict of player...
                                * active
                                * defaultPositionId
                                * droppable
                                * eligibleSlots
                                * firstName
                                * fullName
                                * id
                                * injured
                                * injuryStatus
                                * jersey
                                * lastName
                                * lastNewsDate
                                * proTeamId
                                * stats
                                    - appliedStats
                                    - appliedTotal
                                    - id
                                    - proTeamId
                                    - scoringPeriodId
                                    - seasonId
                                    - statSourceId
                                    - statSplitTypeId
                                    - stats (dict 0-44 keys)
                            - rosterLocked
                            - status
                            - tradeLocked
                        * status
                * rosterForMatchupPeriodDelayed
                    - same as two above, differences unknown as of now
                * teamId
                * tiebreak
                * totalPoints
                * totalPointsLive
            - id
            - matchupPeriodId
            - winner
    - scordingPeriodId
    - seasonId
    - segmentId
    - settings
    - status
    - teams - list
        * abbrev
        * currentProjectedRank
        * divisionId
        * draftDayProjectedRank
        * draftStrategy
        * id
        * isActive
        * logo (url)
        * logoType
        * name
        * owners
        * playoffSeed
        * points
        * pointsAdjusted
        * pointsDelta
        * primaryOwner
        * rankCalculatedFinal
        * rankFinal
        * record
            - away
            - division
            - home
            - overall
        * roster
        * tradeBlock
        * transactionCounter
        * valuesByStat
        * waiverRank



In [109]:
# PARSING LEAGUE'S MEMBERS
team_info_cols = ['currentProjectedRank', 'draftDayProjectedRank', 'id', 'name', 'playoffSeed', 'points']
teams = data['teams']
df = pd.DataFrame(teams)
df = df[team_info_cols]
df = df.rename(columns={'currentProjectedRank' : 'proj_rk' , 'draftDayProjectedRank' : 'draft_proj_rk', 'playoffSeed' : 'seed'})
df = df[['id', 'name', 'points', 'seed', 'proj_rk', 'draft_proj_rk']]
print(df)

   id                           name  points  seed  proj_rk  draft_proj_rk
0   1           Kim Mulkey's Rejects   748.0     2        4              5
1   2  Shyanne Sellars pls come back   525.0     7        5              7
2   3       Quarterzip Memorial Team   592.0     5        6              6
3   4                    Angel REESE   768.0     1        7              8
4   6            Stud bud enthusiast   650.0     4        2              3
5   7          Brenda Frese Fan Club   562.0     6        3              2
6   8            Tatum Tots Top Team   503.0     8        1              1
7   9              Jake's Scary Team   653.0     3        8              4


In [125]:
# PARSING MATCHUPS
# 48 items long (12 weeks, 4 matchups)
# match IDS count from 1-48
schedule = data['schedule']
week1 = schedule[0:4]
for matchup in week1:
    print(matchup['away']['teamId'])

6
2
1
8
